In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

print("✅ Guide Agent libraries imported successfully.")

In [ ]:
print("🔌 Connecting to embeddings & Pinecone Guide Index...")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

GUIDE_INDEX_NAME = "medical-sport-assistant"

# ── Namespace 1: Q&A pairs ──
qa_store = PineconeVectorStore(
    index_name=GUIDE_INDEX_NAME,
    embedding=embeddings,
    namespace="agent_guide_Q&A",
)

# ── Namespace 2: App documentation ──
docs_store = PineconeVectorStore(
    index_name=GUIDE_INDEX_NAME,
    embedding=embeddings,
    namespace="assistant_user_guide",
)

print("✅ Connected to:")
print("   → namespace: agent_guide_Q&A")
print("   → namespace: assistant_user_guide")

In [ ]:
print("🔍 Building Guide Agent retrievers...")

qa_retriever = qa_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 4,
        "score_threshold": 0.50,
    }
)

docs_retriever = docs_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 4,
        "score_threshold": 0.45,
    }
)

def merged_retriever(question: str) -> list:
    seen = set()
    docs = []

    for retriever in [qa_retriever, docs_retriever]:
        for doc in retriever.invoke(question):
            key = hash(doc.page_content[:100])
            if key not in seen:
                seen.add(key)
                docs.append(doc)
    return docs

def format_docs(docs: list) -> str:
    if not docs:
        return "⚠️ No relevant context retrieved from either namespace."
    return "\n\n---\n\n".join([
        f"[Source: {doc.metadata.get('namespace', doc.metadata.get('source', 'unknown'))}]\n"
        f"{doc.page_content}"
        for doc in docs
    ])

print("✅ Multi-namespace retriever ready.")

In [ ]:
print("🧠 Initializing Guide Agent — Llama 3.3 70B...")

guide_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    max_tokens=1024,
    streaming=True,
)

guide_system_prompt = """
You are **SportsMed Guide** — the friendly, knowledgeable onboarding assistant \
for the SportsMed AI application.

Your ONLY job is to help users understand:
  • What the app does and what it cannot do
  • How to use it effectively
  • What topics and domains it covers
  • How to ask better questions
  • How to interpret the answers they receive

You have access to two knowledge sources retrieved below:
  1. ❓ Q&A Knowledge Base  — pre-written answers to common user questions
  2. 📄 App Documentation   — full project description and feature details

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📚 RETRIEVED GUIDE CONTEXT:
{context}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

══════════════════════════════════════════
              GUIDE RULES
══════════════════════════════════════════

▸ RULE 1 — SOURCE STRICT
  Answer ONLY from the retrieved context above.
  Never invent features, capabilities, or instructions
  that are not described in the documentation.

▸ RULE 2 — COVERAGE HANDLING

  ✅ FOUND IN CONTEXT
     Answer clearly, friendly, and structured.

  ⚠️ PARTIALLY FOUND
     Answer what you can, then say:
     "For more details on [specific topic], I recommend
      checking with the app support team."

  ❌ NOT FOUND
     Respond ONLY with:
     "I don't have specific documentation on that yet.
      Try asking about how to use the app, what topics
      it covers, or how to phrase your questions."

▸ RULE 3 — TONE
  • Friendly, clear, and encouraging — not clinical
  • Use simple language — not technical jargon
  • Be concise — users want quick guidance
  • Use examples wherever helpful

▸ RULE 4 — NEVER DO THIS
  • Never answer medical or fitness questions directly
  • Never give training advice or nutrition advice
  • If user asks a medical question → redirect:
    "That sounds like a great question for the
     SportsMed AI assistant! Try asking it in
     the main chat. 💪"

▸ RULE 5 — RESPONSE FORMAT

## 🏷️ [Short clear title]

### Answer
[Direct, friendly answer from context]

### 💡 Tip  ← only when helpful
[Practical suggestion to use the app better]

### 📝 Example  ← only when an example helps
[Show exactly what to type]
"""

guide_prompt = ChatPromptTemplate.from_messages([
    ("system", guide_system_prompt),
    ("human",
     "━━━━━━━━━━━━━━━━━━━━━━━━━\n"
     "👤 USER QUESTION:\n{question}\n"
     "━━━━━━━━━━━━━━━━━━━━━━━━━\n"
     "Check both the Q&A knowledge base and app documentation.\n"
     "Give a clear, friendly, and structured guide response."
    ),
])

print("✅ Guide Agent prompt loaded.")

In [ ]:
print("⛓️ Assembling Guide Agent LCEL chain...")

guide_chain = (
    RunnableParallel({
        "context" : RunnablePassthrough()
                    | (lambda x: merged_retriever(x["question"]))
                    | format_docs,
        "question": RunnablePassthrough()
                    | (lambda x: x["question"]),
    })
    | guide_prompt
    | guide_llm
    | StrOutputParser()
)

def ask_guide(question: str):
    print(f"\n{'━'*60}")
    print(f"👤 User   : {question}")
    print(f"{'━'*60}")

    # ── Show what was retrieved before answering ──
    retrieved_docs = merged_retriever(question)
    print(f"\n📦 Retrieved {len(retrieved_docs)} chunks total")

    if retrieved_docs:
        namespaces = set(
            doc.metadata.get("namespace",
            doc.metadata.get("source", "unknown"))
            for doc in retrieved_docs
        )
        print(f"📁 Namespaces hit : {', '.join(namespaces)}")
    else:
        print("⚠️  No chunks retrieved from either namespace")

    print(f"\n🧭 SportsMed Guide:\n{'─'*60}")

    answer = guide_chain.invoke({"question": question})
    print(answer)
    print(f"{'━'*60}\n")

    return answer

print("✅ Guide Agent LCEL chain ready.")